In [1]:
from os import environ as env
from datetime import datetime
from pyspark.sql import SparkSession

In [2]:
v_file_date = "2024-11-23" # Fecha de corte para carga inicial  

In [3]:
SPARK_CLASSPATH = env['DRIVER_PATH']

In [4]:
# Configuración de SparkSession con soporte S3
spark = SparkSession.builder \
    .appName("ETL Spark") \
    .config("spark.jars", SPARK_CLASSPATH) \
    .config("spark.executor.extraClassPath", SPARK_CLASSPATH) \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", env["MINIO_ROOT_USER"]) \
    .config("spark.hadoop.fs.s3a.secret.key", env["MINIO_ROOT_PASSWORD"]) \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .getOrCreate()

In [5]:
# Conexión a MySQL
jdbc_url = f"jdbc:mysql://mysql:{env['MYSQL_PORT']}/{env['MYSQL_DATABASE']}"
jdbc_properties = {
    "user": env["MYSQL_USER"],
    "password": env["MYSQL_PASSWORD"],
    "driver": "com.mysql.cj.jdbc.Driver"
}

In [8]:
print("\n################## Step 1 - Read data from MySql database ##################\n")
try:
    # Leer una tabla de la base de datos f1db
    sql_query = """
    SELECT *
    FROM f1db.circuits
    """

    df = (
        spark.read.format("jdbc")
        .option("url", jdbc_url)
        .option("driver", jdbc_properties["driver"])
        .option("query", sql_query)
        .option("user", jdbc_properties["user"])
        .option("password", jdbc_properties["password"])
        .load()
    )
    print("\n✅ Connection successful.\n")

except Exception as e:
    print(f"\n❌ Connection failed: {str(e)}")
    raise


################## Step 1 - Read data from MySql database ##################


✅ Connection successful.



In [9]:
df.show(5)

+---------+-----------+--------------------+------------+---------+--------+-------+---+--------------------+
|circuitId| circuitRef|                name|    location|  country|     lat|    lng|alt|                 url|
+---------+-----------+--------------------+------------+---------+--------+-------+---+--------------------+
|        1|albert_park|Albert Park Grand...|   Melbourne|Australia|-37.8497|144.968| 10|http://en.wikiped...|
|        2|     sepang|Sepang Internatio...|Kuala Lumpur| Malaysia| 2.76083|101.738| 18|http://en.wikiped...|
|        3|    bahrain|Bahrain Internati...|      Sakhir|  Bahrain| 26.0325|50.5106|  7|http://en.wikiped...|
|        4|  catalunya|Circuit de Barcel...|    Montmeló|    Spain|   41.57|2.26111|109|http://en.wikiped...|
|        5|   istanbul|       Istanbul Park|    Istanbul|   Turkey| 40.9517| 29.405|130|http://en.wikiped...|
+---------+-----------+--------------------+------------+---------+--------+-------+---+--------------------+
only showi

In [10]:
spark.sparkContext.stop()